In [25]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVR
from sklearn.metrics import mean_absolute_error

In [26]:
DATASET_A_PATH = "bamboo.csv"
DATASET_B_PATH = "duracloud.csv"

SEEDS = [7, 21, 42, 123, 2026]
MAX_FEATURES = 10000

In [27]:
def load_storypoint_dataset(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    required_cols = {"title", "description", "storypoint"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {path}: {missing}")

    df["title"] = df["title"].fillna("").astype(str)
    df["description"] = df["description"].fillna("").astype(str)
    df["storypoint"] = pd.to_numeric(df["storypoint"], errors="coerce")

    df = df.dropna(subset=["storypoint"]).copy()
    df["text"] = (df["title"] + " " + df["description"]).str.strip()

    return df.reset_index(drop=True)

In [28]:
def split_dataset(df: pd.DataFrame, random_state: int):
    train_df, test_df = train_test_split(
        df,
        test_size=0.15,
        random_state=random_state
    )
    return (
        train_df.reset_index(drop=True),
        test_df.reset_index(drop=True)
    )

In [29]:
def build_pipeline(random_state: int) -> Pipeline:
    return Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                stop_words="english",
                ngram_range=(1, 2),
                min_df=2,
                max_features=MAX_FEATURES
            )
        ),
        (
            "regressor",
            LinearSVR(
                random_state=random_state,
                max_iter=10000
            )
        )
    ])

In [30]:
def median_baseline_mae(y_train: np.ndarray, y_test: np.ndarray) -> float:
    median_pred = np.median(y_train)
    preds = np.full(shape=len(y_test), fill_value=median_pred, dtype=float)
    return mean_absolute_error(y_test, preds)


def standardized_accuracy(mae_model: float, mae_baseline: float) -> float:
    if mae_baseline == 0:
        return np.nan
    return 1 - (mae_model / mae_baseline)


def evaluate_model(model, X_test, y_test, y_train_for_baseline):
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    baseline_mae = median_baseline_mae(y_train_for_baseline, y_test)
    sa = standardized_accuracy(mae, baseline_mae)

    return {
        "MAE": mae,
        "Median_Baseline_MAE": baseline_mae,
        "SA": sa
    }

In [31]:
df_a = load_storypoint_dataset(DATASET_A_PATH)
df_b = load_storypoint_dataset(DATASET_B_PATH)

def run_experiment_for_seed(seed: int, df_a: pd.DataFrame, df_b: pd.DataFrame) -> pd.DataFrame:
    train_a, test_a = split_dataset(df_a, random_state=seed)
    train_b, test_b = split_dataset(df_b, random_state=seed)

    # Train on A
    model_a = build_pipeline(random_state=seed)
    model_a.fit(train_a["text"], train_a["storypoint"])

    results_a_to_a = evaluate_model(
        model=model_a,
        X_test=test_a["text"],
        y_test=test_a["storypoint"].values,
        y_train_for_baseline=train_a["storypoint"].values
    )

    results_a_to_b = evaluate_model(
        model=model_a,
        X_test=test_b["text"],
        y_test=test_b["storypoint"].values,
        y_train_for_baseline=train_a["storypoint"].values
    )

    # Train on B
    model_b = build_pipeline(random_state=seed)
    model_b.fit(train_b["text"], train_b["storypoint"])

    results_b_to_b = evaluate_model(
        model=model_b,
        X_test=test_b["text"],
        y_test=test_b["storypoint"].values,
        y_train_for_baseline=train_b["storypoint"].values
    )

    results_b_to_a = evaluate_model(
        model=model_b,
        X_test=test_a["text"],
        y_test=test_a["storypoint"].values,
        y_train_for_baseline=train_b["storypoint"].values
    )

    results_df = pd.DataFrame([
        {"Seed": seed, "Scenario": "A->A", **results_a_to_a},
        {"Seed": seed, "Scenario": "A->B", **results_a_to_b},
        {"Seed": seed, "Scenario": "B->B", **results_b_to_b},
        {"Seed": seed, "Scenario": "B->A", **results_b_to_a},
    ])

    # Delta MAE
    mae_a_within = results_df.loc[results_df["Scenario"] == "A->A", "MAE"].iloc[0]
    mae_a_cross  = results_df.loc[results_df["Scenario"] == "A->B", "MAE"].iloc[0]

    mae_b_within = results_df.loc[results_df["Scenario"] == "B->B", "MAE"].iloc[0]
    mae_b_cross  = results_df.loc[results_df["Scenario"] == "B->A", "MAE"].iloc[0]

    results_df["Delta_MAE_vs_Within"] = np.nan
    results_df.loc[results_df["Scenario"] == "A->B", "Delta_MAE_vs_Within"] = mae_a_cross - mae_a_within
    results_df.loc[results_df["Scenario"] == "B->A", "Delta_MAE_vs_Within"] = mae_b_cross - mae_b_within

    return results_df

In [32]:
all_results = []

for seed in SEEDS:
    seed_results = run_experiment_for_seed(seed, df_a, df_b)
    all_results.append(seed_results)

all_results_df = pd.concat(all_results, ignore_index=True)
all_results_df

,Seed,Scenario,MAE,Median_Baseline_MAE,SA,Delta_MAE_vs_Within
0,7,A->A,1.324841,1.265823,-0.046624,NaN
1,7,A->B,1.090218,1.020000,-0.068841,-0.234623
2,7,B->B,0.900503,0.940000,0.042019,NaN
3,7,B->A,1.340684,1.506329,0.109966,0.440181
4,21,A->A,1.134714,1.126582,-0.007218,NaN
5,21,A->B,1.184163,1.170000,-0.012105,0.049449
6,21,B->B,0.954528,1.070000,0.107918,NaN
7,21,B->A,1.226959,1.367089,0.102502,0.272432
8,42,A->A,1.335138,1.367089,0.023371,NaN
9,42,A->B,1.349127,1.360000,0.007995,0.013989


In [33]:
summary_df = (
    all_results_df
    .groupby("Scenario")[["MAE", "Median_Baseline_MAE", "SA", "Delta_MAE_vs_Within"]]
    .agg(["mean", "std"])
)
summary_df.columns = [
    f"{metric}_{stat}" for metric, stat in summary_df.columns
]
summary_df = summary_df.reset_index()

summary_df

,Scenario,MAE_mean,MAE_std,Median_Baseline_MAE_mean,Median_Baseline_MAE_std,SA_mean,SA_std,Delta_MAE_vs_Within_mean,Delta_MAE_vs_Within_std
0,A->A,1.214184,0.123833,1.220253,0.134737,0.003603,0.038931,NaN,NaN
1,A->B,1.199464,0.131283,1.180000,0.149499,-0.018489,0.029397,-0.014720,0.126966
2,B->A,1.380697,0.162996,1.506329,0.166252,0.083759,0.024969,0.318681,0.085653
3,B->B,1.062016,0.220516,1.176000,0.294160,0.088512,0.051391,NaN,NaN
